# C11-neural-training — Session 3: A NumPy MLP That Learns

*One 90-minute session. Prerequisites: C3's gradient-descent update, C5's MLP
and initialization ideas, and Session 2's complete backward equations.*

**Learning contract.** We will implement the whole
forward → cache → backward → update cycle, train on deterministic tiny data,
plot a loss curve, compare decisions before and after training, and certify one
gradient coordinate numerically.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260804
ATOL = 1e-7
RTOL = 1e-6
rng = np.random.default_rng(SEED)

## 1. Deterministic data and parameter state

We use a repeated XOR-like dataset: $X\in\mathbb R^{N\times2}$ and integer
labels $y\in\{0,1\}^N$. Small seeded Gaussian jitter prevents duplicate
rows while preserving the four clusters. No file or network is involved.

Parameters follow C5's row-is-a-unit convention:
$W_1(H,2)$, $b_1(H)$, $W_2(2,H)$, $b_2(2)$.
The initialization scale $1/\sqrt{d_{in}}$ recalls C5's variance argument.

**Checkpoint 1A.** With $H=8$, state all four parameter shapes.

**Checkpoint 1B.** Why should the same seed reproduce data *and* initial
parameters only when random draws occur in the same order?


In [ ]:
base_X = np.array([[-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0]])
base_y = np.array([0, 1, 1, 0])
X = np.repeat(base_X, 16, axis=0) + 0.12 * rng.normal(size=(64, 2))
y = np.repeat(base_y, 16)
H = 8
params = {
    "W1": rng.normal(scale=1 / np.sqrt(2), size=(H, 2)),
    "b1": np.zeros(H),
    "W2": rng.normal(scale=1 / np.sqrt(H), size=(2, H)),
    "b2": np.zeros(2),
}
print("data:", X.shape, y.shape)
print({name: value.shape for name, value in params.items()})

## 2. Forward pass and cache

A forward pass computes $Z_1=XW_1^\top+b_1$, $A_1=\max(Z_1,0)$,
$Z_2=A_1W_2^\top+b_2$, stable probabilities $P$, and stable mean loss.
For row maximum $m_i$ and shifted logits $S_{ic}=Z_{2,ic}-m_i$, the data
loss is computed directly from logits as

$$L_{\mathrm{data}}=\frac1N\sum_i\left[\log\sum_c e^{S_{ic}}-S_{i,y_i}\right].$$

We cache $P$ for the fused backward gradient, but never compute the loss as
the logarithm of a selected probability that may have underflowed to zero.
The cache is a read-only snapshot for the subsequent backward pass.

For L2 regularization coefficient $\lambda\ge0$, we add
$\frac{\lambda}{2}(\lVert W_1\rVert_F^2+\lVert W_2\rVert_F^2)$.
Biases are deliberately excluded. The matching weight gradient contribution
is $\lambda W$. The main run uses $\lambda=10^{-4}$; setting it to zero
recovers Session 2 exactly.

**Checkpoint 2A.** Why is the regularization loss scalar even though each
weight matrix has many entries?

**Checkpoint 2B.** What inconsistency arises if the forward loss includes L2
but the backward gradients omit $\lambda W$?


In [ ]:
def softmax_and_cross_entropy(logits, labels):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_shifted = np.exp(shifted)
    row_sums = exp_shifted.sum(axis=1, keepdims=True)
    probabilities = exp_shifted / row_sums
    log_normalizers = np.log(row_sums[:, 0])
    correct_shifted = shifted[np.arange(logits.shape[0]), labels]
    loss = (log_normalizers - correct_shifted).mean()
    return probabilities, loss

def forward(X, y, params, l2=0.0):
    Z1 = X @ params["W1"].T + params["b1"]
    A1 = np.maximum(Z1, 0.0)
    Z2 = A1 @ params["W2"].T + params["b2"]
    P, data_loss = softmax_and_cross_entropy(Z2, y)
    reg_loss = 0.5 * l2 * (np.sum(params["W1"]**2) + np.sum(params["W2"]**2))
    cache = (X, y, Z1, A1, P)
    return data_loss + reg_loss, cache

## 3. Backward pass with a shape-preserving return value

Backward starts with $G_2=(P-Y)/N$. Matrix products and reductions exactly
reverse the forward pass. Returning a dictionary with the same keys and
shapes as the parameter dictionary makes the update auditable.

**Checkpoint 3A.** Which two gradients receive L2 contributions?

**Checkpoint 3B.** State a programmatic invariant that catches a transposed
gradient before an update.


In [ ]:
def backward(cache, params, l2=0.0):
    X, y, Z1, A1, P = cache
    n = X.shape[0]
    G2 = P.copy()
    G2[np.arange(n), y] -= 1.0
    G2 /= n
    dW2 = G2.T @ A1 + l2 * params["W2"]
    db2 = G2.sum(axis=0)
    G1 = (G2 @ params["W2"]) * (Z1 > 0.0)
    dW1 = G1.T @ X + l2 * params["W1"]
    db1 = G1.sum(axis=0)
    grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
    assert all(grads[name].shape == params[name].shape for name in params)
    return grads

initial_loss, initial_cache = forward(X, y, params, l2=1e-4)
initial_grads = backward(initial_cache, params, l2=1e-4)
print("initial loss:", initial_loss)
print({name: grad.shape for name, grad in initial_grads.items()})

## 4. Update ordering and a complete training loop

One full-batch step is:

1. forward with the current parameters;
2. backward from that exact cache;
3. update *all* parameters with
   $\theta\leftarrow\theta-\eta\nabla_\theta L$.

Here $\eta$ is the learning rate. We log loss before each update. The loop is
full-batch gradient descent; a mini-batch loop differs only by selecting a
seeded subset before forward. Never update one layer while still computing
another layer's gradient.

**Checkpoint 4A.** Why does `new = old - learning_rate * gradient` require
the gradient and parameter to have identical shapes?

**Checkpoint 4B.** If loss rises immediately, name two quantities to audit
before declaring the task impossible.


In [ ]:
def train(X, y, initial_params, *, steps, learning_rate, l2):
    trained = {name: value.copy() for name, value in initial_params.items()}
    losses = []
    for _ in range(steps):
        loss, cache = forward(X, y, trained, l2=l2)
        grads = backward(cache, trained, l2=l2)
        losses.append(loss)
        for name in trained:
            trained[name] -= learning_rate * grads[name]
    return trained, np.array(losses)

untrained = {name: value.copy() for name, value in params.items()}
trained, losses = train(
    X, y, untrained, steps=1200, learning_rate=0.12, l2=1e-4
)
print("first/final loss:", losses[0], losses[-1])
assert np.isfinite(losses).all()
assert losses[-1] < 0.08
assert losses[-1] < losses[0]

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(losses)
ax.set(xlabel="gradient step", ylabel="mean CE + L2", title="Seeded NumPy MLP training")
ax.grid(alpha=0.25)
plt.show()

## 5. Training certification: decisions changed for a reason

A low final loss alone is insufficient. A **trained-MLP certificate** checks:

- finite loss and a meaningful reduction from the initial loss;
- parameter movement from initialization;
- class decisions on named inputs;
- deterministic reproduction under the same seed and draw order.

The decision is `argmax` across the class axis of logits/probabilities,
therefore output shape $(N,)$. We compare the four cluster centers before and
after training.

**Checkpoint 5A.** Why is comparing trained weights to one hard-coded array a
fragile certificate?

**Checkpoint 5B.** Can accuracy improve while cross-entropy worsens? Explain
briefly.


In [ ]:
def predict(X, params):
    dummy_y = np.zeros(X.shape[0], dtype=int)
    _loss, (_X, _y, _Z1, _A1, P) = forward(X, dummy_y, params)
    return P.argmax(axis=1), P

before_pred, before_prob = predict(base_X, untrained)
after_pred, after_prob = predict(base_X, trained)
movement = {
    name: float(np.linalg.norm(trained[name] - untrained[name]))
    for name in trained
}
print("targets:", base_y)
print("before:", before_pred, "\nafter: ", after_pred)
print("movement:", movement)
assert np.array_equal(after_pred, base_y)
assert max(movement.values()) > 0.1

## 6. Gradient check and worked debugging audit

**Worked exam-register example 2.** A trainer reports decreasing loss, but
`W1` never changes. Inspection shows:

1. `dW1.shape == W1.shape`;
2. `np.linalg.norm(dW1) > 0`;
3. the update loop iterates over `["W2", "b2"]` only.

The forward and backward mathematics are not the fault. The parameter
registry for the update omits `W1` and `b1`. The minimal fix is to iterate
over all keys in the parameter dictionary, and then certify nonzero movement
for each expected trainable group. This is a **training/debug audit**:
evidence identifies the broken lifecycle stage.

We separately finite-difference one $W_1$ coordinate. The chosen point is away
from ReLU zero; $\varepsilon=10^{-6}$; equality uses the displayed tolerances.

**Checkpoint 6A.** If the numeric and analytic gradients disagree only for
biases by factor $N$, locate the likely bug.

**Checkpoint 6B.** Why should the gradient check run before a long training
loop during development?


In [ ]:
check_params = {name: value.copy() for name, value in untrained.items()}
check_loss, check_cache = forward(X[:8], y[:8], check_params, l2=1e-4)
check_grads = backward(check_cache, check_params, l2=1e-4)
epsilon = 1e-6
r, c = 0, 0
plus = {name: value.copy() for name, value in check_params.items()}
minus = {name: value.copy() for name, value in check_params.items()}
plus["W1"][r, c] += epsilon
minus["W1"][r, c] -= epsilon
loss_plus, _ = forward(X[:8], y[:8], plus, l2=1e-4)
loss_minus, _ = forward(X[:8], y[:8], minus, l2=1e-4)
numeric = (loss_plus - loss_minus) / (2 * epsilon)
analytic = check_grads["W1"][r, c]
print("numeric:", numeric, "analytic:", analytic)
assert np.isclose(numeric, analytic, atol=ATOL, rtol=RTOL)

## 7. Pitfalls, regularization links, and exam connections

- **Stale cache:** forward, update, then backward differentiates the wrong
  function state. Always backward before any update.
- **Aliased initial state:** `trained = initial` mutates the baseline too.
  Copy every array before training.
- **Unseeded mini-batches:** loss curves then cannot be reproduced. Own one
  generator and record its seed.
- **Regularization mismatch:** the $lambda/2$ forward convention must pair
  with $lambda W$ backward.
- **Accuracy-only audit:** `argmax` can stay fixed while probabilities and
  loss change substantially.

Round 1 coding tasks may provide the helper functions and ask you to repair one
axis, cache, or update. Write the lifecycle and shape table before tracing
values. Session 4 maps each NumPy stage to PyTorch autograd and optimizers.

**Checkpoint 7A.** Why should evaluation avoid adding L2 to the reported
classification accuracy?

**Checkpoint 7B.** What three artifacts make this tiny run reproducible?


## Checkpoint answers

**1A.** $W_1(8,2),b_1(8,),W_2(2,8),b_2(2,)$. **1B.** A seed fixes the
generator stream, not semantic names; reordering draws assigns different
stream values.

**2A.** It sums all squared scalar entries. **2B.** The returned gradient is
not the derivative of the returned loss.

**3A.** $dW_1,dW_2$. **3B.** For every key,
`grads[key].shape == params[key].shape`.

**4A.** The update is coordinatewise and must preserve parameter structure.
**4B.** Learning rate and gradient scale/sign (then shapes and averaging).

**5A.** Equivalent valid training runs need not land at identical weights.
**5B.** Yes; hard decisions can improve while confidence on other examples
falls enough to increase log loss.

**6A.** Bias contributions were averaged after $G_2$ was already divided by
$N$. **6B.** It isolates derivative bugs before parameter motion and long-loop
effects obscure them.

**7A.** L2 is a training objective term, not a count of correct labels.
**7B.** Fixed data-generation seed and order, copied initial parameters, and
fixed loop hyperparameters/order.
